In [8]:
import torch
import gc

def real_cleanup():
    # 1. Delete all references to GPU tensors and models
    global model, optimizer, data  # Reference your variables here
    
    if 'model' in globals():
        del model
    if 'optimizer' in globals():
        del optimizer
    if 'data' in globals():
        del data
    
    # 2. Clear any other GPU tensors
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                del obj
        except:
            pass
    
    # 3. Force garbage collection
    gc.collect()
    
    # 4. Clear CUDA cache (now actually effective)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# Usage:
real_cleanup()

/home/guests/andreea_magureanu/.local/lib/python3.11/site-packages/torch/__init__.py:1125: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [ ]:
def comprehensive_cleanup():
    """Most effective GPU memory cleanup for model reloading"""
    
    # 1. Delete the specific model and related objects
    if 'model' in globals():
        del model
    if 'tokenizer' in globals():
        del tokenizer
    
    # 2. Clear any other GPU tensors by removing references
    for obj in list(gc.get_objects()):
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                del obj
        except:
            pass
    
    # 3. Force garbage collection
    gc.collect()
    
    # 4. Clear CUDA cache aggressively
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()  # Wait for operations to complete

# Usage before loading MedGemma:
comprehensive_cleanup()

In [ ]:
# Check what processes are using GPU memory
import subprocess
import torch

def get_gpu_processes():
    try:
        result = subprocess.check_output(
            ['nvidia-smi', '--query-compute-apps=pid,process_name,used_memory', '--format=csv,noheader'],
            encoding='utf-8'
        )
        print("=== GPU Processes ===")
        print("PID | Process Name | Memory Used")
        print("-" * 40)
        for line in result.strip().split('\n'):
            if line.strip():
                print(line)
    except Exception as e:
        print(f"Error getting GPU processes: {e}")

get_gpu_processes()

# Also check CUDA memory details
print("\n=== CUDA Memory Details ===")
torch.cuda.empty_cache()  # Try to clear cache first
for i in range(torch.cuda.device_count()):
    print(f"\nGPU {i}:")
    print(f"Memory allocated: {torch.cuda.memory_allocated(i) / 1024**2:.1f} MB")
    print(f"Memory reserved: {torch.cuda.memory_reserved(i) / 1024**2:.1f} MB")
    print(f"Max memory allocated: {torch.cuda.max_memory_allocated(i) / 1024**2:.1f} MB")

=== GPU Processes ===
PID | Process Name | Memory Used
----------------------------------------
735453, /home/guests/andreea_magureanu/.conda/envs/rare_dis/bin/python, 8822 MiB
736617, /home/guests/andreea_magureanu/.conda/envs/rare_dis/bin/python, 254 MiB

=== CUDA Memory Details ===

GPU 0:
Memory allocated: 0.0 MB
Memory reserved: 0.0 MB
Max memory allocated: 8213.3 MB


: 

In [1]:
import requests
from lxml import etree as ET

PMCID = "PMC9314610"

URLS = [
    f"https://www.ebi.ac.uk/europepmc/webservices/rest/{PMCID}/fullTextXML",
    f"https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi?verb=GetRecord&identifier=oai:pubmedcentral.nih.gov:{PMCID}&metadataPrefix=pmc",
]

DROP_HEADS = {
    "acknowledgements","acknowledgments","funding","funding information",
    "conflict of interest","competing interests","author contributions",
    "data availability","ethics","references","bibliography",
    "supplementary","appendix","correspondence"
}

def fetch_xml():
    last_err = None
    for url in URLS:
        try:
            r = requests.get(url, timeout=90)
            r.raise_for_status()
            # Quick sanity check: must contain an <article> element
            if b"<article" in r.content:
                return ET.fromstring(r.content)
            # Some wrappers (OAI) put <article> deeper; still okay
            try:
                root = ET.fromstring(r.content)
                if root.xpath("//*[local-name()='article']"):
                    return root
            except ET.XMLSyntaxError as e:
                last_err = e
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to fetch XML for {PMCID}: {last_err}")

def norm_text(node):
    return " ".join(" ".join(node.itertext()).split())

def parse_title_abs_paras(root):
    # Find the article node regardless of namespaces/wrappers
    article_nodes = root.xpath("//*[local-name()='article']")
    if not article_nodes:
        return "", "", []
    article = article_nodes[0]

    # Title
    title_nodes = article.xpath(".//*[local-name()='article-title']")
    title = norm_text(title_nodes[0]) if title_nodes else ""

    # Abstract (join all parts)
    abs_nodes = article.xpath(".//*[local-name()='abstract']")
    abstract = norm_text(abs_nodes[0]) if abs_nodes else ""

    # Body → sections → paragraphs (skip unwanted heads)
    paras = []
    for sec in article.xpath(".//*[local-name()='body']//*[local-name()='sec']"):
        head = " ".join(sec.xpath(".//*[local-name()='title']/text()")).strip().lower()
        if any(bad in head for bad in DROP_HEADS):
            continue
        for p in sec.xpath(".//*[local-name()='p']"):
            t = norm_text(p)
            if len(t.split()) >= 5:
                paras.append(t)

    # Fallback: any <p> under body if secs were missed
    if not paras:
        for p in article.xpath(".//*[local-name()='body']//*[local-name()='p']"):
            t = norm_text(p)
            if len(t.split()) >= 5:
                paras.append(t)

    return title, abstract, paras

if __name__ == "__main__":
    root = fetch_xml()
    title, abstract, paras = parse_title_abs_paras(root)

    print("\nTITLE:\n", title, "\n")
    print("ABSTRACT (first 500 chars):\n", abstract[:500], "...\n")
    print("FIRST 3 PARAGRAPHS:\n")
    for i, p in enumerate(paras[:3], 1):
        print(f"[{i}] {p}\n")
    print(f"Total paragraphs found: {len(paras)}")



TITLE:
 Genotype–phenotype correlates in Joubert syndrome: A review 

ABSTRACT (first 500 chars):
 Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,” and variable organ involvement. Over 40 causative genes have been identified to date, explaining up to 94% of cases. To date, gene‐phenotype correlates have been delineated only for a handful of genes, directly translating into improved counseling and clinical care. For instance, JS individuals harboring pathogenic varian ...

FIRST 3 PARAGRAPHS:

[1] Joubert syndrome (JS) is a rare congenital neurodevelopmental primary ciliopathy with a population‐based prevalence reaching 1.7 per 100,000 in the age range 0–19 years (Nuovo et al., 2020 ). First described by Dr Marie Joubert about 50 years ago (Joubert, Eisenring, Robb, & Andermann, 1969 ), JS is now diagnosed upon recognition of a pathognomonic malformation of th

In [13]:
print(abstract)

Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,” and variable organ involvement. Over 40 causative genes have been identified to date, explaining up to 94% of cases. To date, gene‐phenotype correlates have been delineated only for a handful of genes, directly translating into improved counseling and clinical care. For instance, JS individuals harboring pathogenic variants in TMEM67 have a significantly higher risk of liver fibrosis, while pathogenic variants in NPHP1 , RPGRIP1L , and TMEM237 are frequently associated to JS with renal involvement, requiring a closer monitoring of liver parameters, or renal functioning. On the other hand, individuals with causal variants in the CEP290 or AHI1 need a closer surveillance for retinal dystrophy and, in case of CEP290 , also for chronic kidney disease. These examples highlight how an accurate description of the range

In [2]:
import gc, os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"

def _cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def _is_oom(e: Exception) -> bool:
    m = str(e).lower()
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in m or "cuda oom" in m


tok = AutoTokenizer.from_pretrained(
    MODEL_DIR, local_files_only=True, use_fast=True, trust_remote_code=True
)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

try:
    _cleanup()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        local_files_only=True,
        trust_remote_code=True,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    ).to("cuda")
    mode = f"fp{16 if dtype==torch.float16 else 'bf16'}"
    print("Loaded model ok - using dtype:", dtype)

except Exception as e:
    if not _is_oom(e):
        raise  

    print("OOM on fp16/bf16 load. Falling back to bitsandbytes 4-bit…")
    _cleanup()

    try:
       
        bnb4 = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16),
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_DIR,
            trust_remote_code=True,
            local_files_only=True,
            device_map="auto",         
            quantization_config=bnb4,
            low_cpu_mem_usage=True,
        )
        mode = "4bit_nf4_auto"
        print("Loaded with bitsandbytes 4-bit (NF4).")

    except Exception as e4:
        if _is_oom(e4):
            print("Still OOM on 4-bit. Trying 8-bit…")
        else:
            print(f"4-bit failed ({type(e4).__name__}: {e4}). Trying 8-bit…")
        _cleanup()
        try:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_DIR,
                trust_remote_code=True,
                local_files_only=True,
                device_map="auto",
                load_in_8bit=True,       
                low_cpu_mem_usage=True,
            )
            mode = "8bit_auto"
            print("Loaded with bitsandbytes 8-bit.")
        except Exception as e8:
          
            print(f"8-bit failed ({type(e8).__name__}: {e8}). Trying auto offload…")
            _cleanup()
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_DIR,
                trust_remote_code=True,
                local_files_only=True,
                device_map="auto",
                max_memory={0: "90%", "cpu": "64GiB"},  # avoid crash, spill to CPU
                torch_dtype="auto",
                low_cpu_mem_usage=True,
            )
            mode = "half_auto_offload"
            print("Loaded with auto offload (GPU+CPU).")

print("Mode:", mode)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded model ok - using dtype: torch.bfloat16
Mode: fpbf16


In [ ]:
import re, json
def chat(messages, max_new_tokens=512):
    device = "cuda"
    prompt_text = tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # mark where assistant should start
    )

    enc = tok(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    )

    enc = {k: v.to(device) for k, v in enc.items()}


    out_ids = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,                 # start deterministic
        eos_token_id=tok.eos_token_id,   # clean stop
        pad_token_id=tok.pad_token_id,   # silence warnings
        )

    # Decode only the new tokens (no prompt echo)
    new_tokens = out_ids[0, enc["input_ids"].shape[-1]:]
    out_text = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return out_text

In [4]:
# Best VARIANT FOR ENTITIES!!!!!

system_step1 = (
    "You are a precise medical information extraction agent. Your task is to identify and normalize medical entities from the provided text. "
    "Output MUST be a SINGLE JSON object with ONLY an 'entities' key, whose value is a list of objects. "
    "Each entity object must have: "
    "'name' (string, the primary/canonical name as used in the text), "
    "'type' (string, must be one of: 'rare_disease', 'phenotype', 'gene', 'genotype', 'treatment'), "
   "If an entity is mentioned multiple times, keep just the first occurrence "
    "Do not output any relations. Do not output any other keys or explanatory text."
)
user_step1 = (
    f"Title: {title}\n"
    f"Abstract: {abstract}\n\n"
    "Task: Extract entities of types rare_disease, phenotype, gene, genotype, treatment, "
    "with explicit aliases only. Output the JSON now."
)

messages_entity = [
    {"role": "system", "content": system_step1},
    {"role": "user",   "content": user_step1},
]

In [5]:
entities = chat(messages_entity)
print(entities)

```json
{
  "entities": [
    {
      "name": "Joubert syndrome",
      "type": "rare_disease"
    },
    {
      "name": "cerebellar and brainstem malformation",
      "type": "phenotype"
    },
    {
      "name": "molar tooth sign",
      "type": "phenotype"
    },
    {
      "name": "liver fibrosis",
      "type": "phenotype"
    },
    {
      "name": "renal involvement",
      "type": "phenotype"
    },
    {
      "name": "retinal dystrophy",
      "type": "phenotype"
    },
    {
      "name": "chronic kidney disease",
      "type": "phenotype"
    },
    {
      "name": "TMEM67",
      "type": "gene"
    },
    {
      "name": "NPHP1",
      "type": "gene"
    },
    {
      "name": "RPGRIP1L",
      "type": "gene"
    },
    {
      "name": "TMEM237",
      "type": "gene"
    },
    {
      "name": "CEP290",
      "type": "gene"
    },
    {
      "name": "AHI1",
      "type": "gene"
    }
  ]
}
```


In [6]:
def extract_entity_names(text: str) -> list:
    """
    Extract just the entity names from the model output, even if JSON is malformed.
    Returns a list of entity names.
    """
    names = set()
    
    # Try to find any JSON-like structure first
    json_matches = re.finditer(r'\{[^{}]*\}', text)
    
    for match in json_matches:
        try:
            data = json.loads(match.group(0))
            if 'entities' in data and isinstance(data['entities'], list):
                for entity in data['entities']:
                    if isinstance(entity, dict) and 'name' in entity:
                        names.add(entity['name'])
        except json.JSONDecodeError:
            continue
    
    # If no JSON found, try to extract names directly using regex patterns
    if not names:
        # Look for name patterns like "name": "something"
        name_matches = re.finditer(r'"name"\s*:\s*"([^"]+)"', text)
        for match in name_matches:
            names.add(match.group(1))
    
    return names

# Test with your last output
print("Extracting entity names from output:")
entity_names = extract_entity_names(entities)
print("\nFound entity names:")
for name in entity_names:
    print(f"- {name}")

# These names can now be used in your next prompt for alias extraction

Extracting entity names from output:

Found entity names:
- TMEM67
- AHI1
- CEP290
- cerebellar and brainstem malformation
- NPHP1
- Joubert syndrome
- renal involvement
- chronic kidney disease
- retinal dystrophy
- liver fibrosis
- TMEM237
- RPGRIP1L
- molar tooth sign


In [7]:
print(entity_names)
print(type(entity_names))
import json 
# Convert set to list before JSON serialization
entity_names_json = json.dumps(list(entity_names), ensure_ascii=False)
print(entity_names_json)

{'TMEM67', 'AHI1', 'CEP290', 'cerebellar and brainstem malformation', 'NPHP1', 'Joubert syndrome', 'renal involvement', 'chronic kidney disease', 'retinal dystrophy', 'liver fibrosis', 'TMEM237', 'RPGRIP1L', 'molar tooth sign'}
<class 'set'>
["TMEM67", "AHI1", "CEP290", "cerebellar and brainstem malformation", "NPHP1", "Joubert syndrome", "renal involvement", "chronic kidney disease", "retinal dystrophy", "liver fibrosis", "TMEM237", "RPGRIP1L", "molar tooth sign"]


In [ ]:
#GEMINI
import langextract as lx
import textwrap




prompt = textwrap.dedent(f"""\
    Extract alias relationships for the given biomedical entities.
    Focus on instances where a canonical entity is mentioned with an alternate name 
    (e.g., "also known as ___" or "previously called ___", "aka", "canonical name (aliase)").
    The known canonical entities include: {", ".join(entity_names_json)}.
    For each alias found, use the exact text of the alias and pair it with the canonical name.
    Do not infer aliases that are not explicitly stated in the text.
""")

examples = [
    lx.data.ExampleData(
        text="Joubert syndrome (previously called vermian aplasia) is a rare genetic condition.",
        extractions=[
            lx.data.Extraction(
                extraction_class="alias",
                extraction_text="vermian aplasia",
                attributes={"canonical": "Joubert syndrome"}
            )
        ]
    ),
    lx.data.ExampleData(
        text="Mutations in the CEP290 gene (also known as NPHP6) are a common cause of Joubert syndrome.",
        extractions=[
            lx.data.Extraction(
                extraction_class="alias",
                extraction_text="NPHP6",
                attributes={"canonical": "CEP290"}
            )
        ]
    )
]

result = lx.extract(
    text_or_documents=abstract,
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash",                    
    
    fence_output=True, 
    use_schema_constraints=False
)



In [11]:

aliases = []
for ext in result.extractions: 
    if ext.extraction_class == "alias":
        canonical = ext.attributes.get("canonical")
        alias_name = ext.extraction_text
        aliases.append((canonical, alias_name))
        print(f"Alias found: {canonical} -> {alias_name}")




Alias found: Joubert syndrome -> JS
Alias found: cerebellar and brainstem malformation -> molar tooth sign
